## Mini Data Quality Audit

**Dataset:** week2_customer_transactions_messy.csv  


In [3]:
from google.colab import files
uploaded = files.upload()

Saving week2_customer_transactions_messy.csv to week2_customer_transactions_messy (1).csv


In [4]:
import pandas as pd

# Load dataset (adjust path if needed)
df = pd.read_csv('week2_customer_transactions_messy.csv')

# Preview data
df.head()

,transaction_id,customer_id,transaction_date,amount,currency,payment_method,status,region,last_updated
0,T0001,C100,2026-01-05,120.50,EUR,card,completed,DE,2026-01-05
1,T0002,C101,2026/01/06,0.00,EUR,CARD,completed,de,2026-01-20
2,T0003,C102,06-01-2026,-35.00,USD,bank_transfer,completed,US,2026-01-07
3,T0004,NaN,2026-01-07,250.00,EUR,card,pending,FR,2026-01-08
4,T0005,C104,2026-01-07,89.99,EURO,cash,completed,DE,2026-01-09


##  Dataset Description

This dataset contains customer transaction records, including fields such as:
- Customer ID
- Transaction ID
- Transaction Date
- Product/Category
- Transaction Amount
- Payment Method

###  Business Use Case
The dataset can be used for:
- Sales analysis
- Customer behavior insights
- Revenue tracking
- Fraud detection

Poor data quality in this dataset can lead to incorrect business decisions and unreliable reporting.

In [8]:
df.info()
df.describe(include='all')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11 entries, 0 to 10
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   transaction_id    11 non-null     object 
 1   customer_id       10 non-null     object 
 2   transaction_date  11 non-null     object 
 3   amount            10 non-null     float64
 4   currency          11 non-null     object 
 5   payment_method    10 non-null     object 
 6   status            11 non-null     object 
 7   region            11 non-null     object 
 8   last_updated      10 non-null     object 
dtypes: float64(1), object(8)
memory usage: 924.0+ bytes


,transaction_id,customer_id,transaction_date,amount,currency,payment_method,status,region,last_updated
count,11,10,11,10.000000,11,10,11,11,10
unique,10,9,8,NaN,3,4,3,5,9
top,T0006,C105,2026-01-07,NaN,EUR,card,completed,DE,2026-04-15
freq,2,2,2,NaN,9,6,8,6,2
mean,NaN,NaN,NaN,100056.457000,NaN,NaN,NaN,NaN,NaN
std,NaN,NaN,NaN,316207.939002,NaN,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,-35.000000,NaN,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,19.990000,NaN,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,49.550000,NaN,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,112.872500,NaN,NaN,NaN,NaN,NaN


##  Data Quality Assessment

### 1. Completeness
- Missing customer_id detected

### 2. Validity
- Negative transaction amount (-35)
- Zero transaction amount (0)

### 3. Consistency
- payment_method inconsistency ("card", "CARD")
- currency inconsistency ("EUR", "EURO")
- region inconsistency ("DE", "de")

### 4. Format Issues
- transaction_date has mixed formats:
  - YYYY-MM-DD
  - YYYY/MM/DD
  - DD-MM-YYYY

### 5. Integrity
- Missing customer_id affects relational integrity

In [9]:
# Completeness Rate
completeness_rate = df.notnull().sum().sum() / (df.shape[0] * df.shape[1])

# Duplication Rate
duplication_rate = df.duplicated().sum() / len(df)

# Error Rate (invalid amounts)
error_rate = ((df['amount'] <= 0)).sum() / len(df)

print("Completeness Rate:", round(completeness_rate, 2))
print("Duplication Rate:", round(duplication_rate, 2))
print("Error Rate:", round(error_rate, 2))

Completeness Rate: 0.96
Duplication Rate: 0.09
Error Rate: 0.18


##  KPI Interpretation

- **Completeness Rate:** High, but at least one missing customer_id exists.
- **Duplication Rate:** Low or zero (no duplicate rows detected).
- **Error Rate:** Non-zero due to negative and zero transaction amounts.

This indicates moderate data quality issues mainly related to validity and consistency.

In [10]:
# Rule 1: Amount must be positive
invalid_amounts = df[df['amount'] <= 0]

# Rule 2: Customer ID must not be missing
missing_customer = df[df['customer_id'].isnull()]

# Rule 3: Standardize and validate payment methods
valid_methods = ['card', 'cash', 'bank_transfer']
invalid_payment = df[~df['payment_method'].str.lower().isin(valid_methods)]

# Rule 4: Detect inconsistent currency
valid_currency = ['EUR', 'USD']
invalid_currency = df[~df['currency'].isin(valid_currency)]

print("Invalid amounts:", len(invalid_amounts))
print("Missing customer_id:", len(missing_customer))
print("Invalid payment methods:", len(invalid_payment))
print("Invalid currency:", len(invalid_currency))

Invalid amounts: 2
Missing customer_id: 1
Invalid payment methods: 1
Invalid currency: 1


##  Audit Summary

| Issue Type            | Affected Rows | Severity | Recommended Action |
|----------------------|-------------|----------|-------------------|
| Missing customer_id  | 1           | High     | Remove or impute |
| Invalid amounts      | 2           | High     | Remove or correct |
| Inconsistent payment | Multiple    | Medium   | Standardize case |
| Inconsistent currency| 1           | Medium   | Convert "EURO" → "EUR" |
| Date format issues   | Multiple    | Medium   | Standardize format |
| Region inconsistency | Multiple    | Low      | Convert to uppercase |

##  Recommended Cleaning Actions

- Handle missing values:
  - Remove or impute missing customer_id

- Fix numerical issues:
  - Remove or correct negative/zero amounts

- Standardize categorical values:
  - payment_method → lowercase
  - currency → standard codes (EUR, USD)
  - region → uppercase

- Fix date formats:
  - Convert all to YYYY-MM-DD

- Validate schema:
  - Ensure correct data types for each column

##  Final Summary

The dataset contains several data quality issues affecting validity, consistency, and completeness.

Key findings:
- Invalid transaction amounts
- Missing customer identifiers
- Inconsistent categorical values
- Mixed date formats

These issues must be addressed before performing reliable analysis.